# AFib detector — two improvements tested on unseen data (Notebook 5)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebooks 3–4 found two weaknesses on the external PhysioNet/CinC 2017 data:
1. **43 % of the 30-second recordings are too short** for the 32-beat model (it needs ≥ 33 beats).
2. **Motion noise raises false AF calls**, and the app's signal-quality index does not catch it.

Two fixes, each with rules fixed before the run and tested on the same external data as before:

**(a) A 16-beat model** — same architecture and training as the 32-beat model, only the input is 16 RR intervals.
Training windows are taken every 4 beats instead of 8 so that the two models see a similar number of windows.
Compared with the 32-beat model on (i) the recordings both can classify, (ii) all AF/Normal recordings, counting
"cannot classify" as "no AF".

**(b) Beat-detector agreement (bSQI)** — the idea of Li, Mark and Clifford (2008): run two independent beat
detectors; where the signal is clean they agree, where it is noisy they disagree. Second detector: `wfdb` XQRS.
For every 32-beat decision, bSQI of the 10 s before it = matched beats / (beats₁ + beats₂ − matched),
match tolerance 150 ms; the decision is dropped if bSQI < 0.8. A recording with no decision left gets no verdict.
Same noise protocol as Notebook 4 (NSTDB electrode motion and muscle noise at 12/6/0 dB, same seed).

> Kaggle: *Add Input* → `marufhs/atrial-fibrillation`; Internet **On**.

In [ ]:
import os, glob, time, random, contextlib, io, json, subprocess, sys, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score

SEED = 42
FS_ORIG, FS = 250, 125
N_RR, RR_STRIDE, DEV_STRIDE = 32, 8, 4
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
LABELS = {'(N': 0, '(AFIB': 1}
tf.get_logger().setLevel('ERROR')
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

AFDB_DIR = os.environ.get('AFDB_DIR') or [d for d in {os.path.dirname(p) for p in glob.glob('/kaggle/input/**/04015.dat', recursive=True)} if '/old' not in d][0]
records = sorted(r[:-4] for r in os.listdir(AFDB_DIR) if r.endswith('.dat') and os.path.exists(os.path.join(AFDB_DIR, r[:-4] + '.atr')))
print(len(records), 'AFDB records')

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def rr_features(rr):
    rr = np.clip(rr, 0.2, 2.0)
    return np.stack([(rr - 0.8) / 0.25, (rr / rr.mean() - 1) / 0.15], -1).astype(np.float32)

def to_125(sig, fs):
    # same band-pass as the firmware (0.5-40 Hz, 2nd order), then to 125 Hz
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    if fs == 250: return x[::2]
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

In [ ]:
def tcn_block(x, f, k, d, drop):
    sc = x
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d)(x)
    if sc.shape[-1] != f: sc = layers.Conv1D(f, 1)(sc)
    return layers.ReLU()(layers.Add()([x, sc]))

def build_rr_tcn():
    inp = keras.Input(shape=(N_RR, 2)); x = inp
    for d in (1, 2, 4, 8): x = tcn_block(x, 16, 3, d, 0.1)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(layers.GlobalAveragePooling1D()(x)))

def to_int8(m, Xrep):
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    rep = Xrep[np.random.RandomState(0).choice(len(Xrep), min(1000, len(Xrep)), replace=False)]
    conv.representative_dataset = lambda: ([s[None]] for s in rep)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return conv.convert()

def run_tflite(blob, Xd, bs=4096):
    outs = []
    it = tf.lite.Interpreter(model_content=blob)
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]; inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], [len(xb), N_RR, 2]); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        sc, zp = inp['quantization']; it.set_tensor(inp['index'], np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8))
        it.invoke(); sc, zp = out['quantization']
        outs.append(((it.get_tensor(out['index']).astype(np.float32) - zp) * sc).ravel())
    return np.concatenate(outs) if outs else np.zeros(0)

PREP = {}
for rec in records:
    sig = wfdb.rdrecord(os.path.join(AFDB_DIR, rec), channels=[0]).p_signal[:, 0]
    ann = wfdb.rdann(os.path.join(AFDB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    segs = [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
    x = to_125(sig, FS_ORIG); beats = detect_r(x); rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    Xw, yw = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE):
        labs = lab_at[beats[k]:beats[k + N_RR]]
        if len(labs) == 0 or labs.min() < 0 or labs.min() != labs.max(): continue
        Xw.append(rr_features(rr[k:k + N_RR])); yw.append(int(labs[0]))
    PREP[rec] = (np.array(Xw, np.float32).reshape(-1, N_RR, 2), np.array(yw, np.int32))

rng = np.random.RandomState(SEED)
for _ in range(500):   # 2 patients only for early stopping, both rhythms present (same rule as Notebook 2)
    val = list(rng.choice(records, 2, replace=False))
    if all(sum((PREP[v][1] == c).sum() for v in val) >= 100 for c in (0, 1)): break
train = [r for r in records if r not in val]
Xtr = np.concatenate([PREP[r][0] for r in train]); ytr = np.concatenate([PREP[r][1] for r in train])
Xva = np.concatenate([PREP[r][0] for r in val]);   yva = np.concatenate([PREP[r][1] for r in val])
cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * ytr.sum())}
model = build_rr_tcn(); model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
h = model.fit(ds, validation_data=(Xva, yva), epochs=int(os.environ.get('EPOCHS', 40)), class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])
BLOB = to_int8(model, Xtr)
open(f'{OUT_DIR}/afib_rr_tcn_all23_int8.tflite', 'wb').write(BLOB)
print('trained on', len(train), 'patients (+2 for early stopping:', val, '), epochs', len(h.history['loss']),
      '| windows', len(ytr), '| int8 model', len(BLOB), 'bytes')

## (a) Train the 16-beat model the same way

In [ ]:
def run_tflite(blob, Xd, bs=4096):       # shape taken from the data, so it serves both models
    outs = []; it = tf.lite.Interpreter(model_content=blob)
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]; inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], list(xb.shape)); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        sc, zp = inp['quantization']; it.set_tensor(inp['index'], np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8))
        it.invoke(); sc, zp = out['quantization']
        outs.append(((it.get_tensor(out['index']).astype(np.float32) - zp) * sc).ravel())
    return np.concatenate(outs) if outs else np.zeros(0)

BLOB32 = BLOB
N_RR = 16
PREP16 = {}
for rec in records:
    sig = wfdb.rdrecord(os.path.join(AFDB_DIR, rec), channels=[0]).p_signal[:, 0]
    ann = wfdb.rdann(os.path.join(AFDB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    segs = [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
    x = to_125(sig, FS_ORIG); beats = detect_r(x); rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    Xw, yw = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE // 2):
        labs = lab_at[beats[k]:beats[k + N_RR]]
        if len(labs) == 0 or labs.min() < 0 or labs.min() != labs.max(): continue
        Xw.append(rr_features(rr[k:k + N_RR])); yw.append(int(labs[0]))
    PREP16[rec] = (np.array(Xw, np.float32).reshape(-1, N_RR, 2), np.array(yw, np.int32))
# same 2 early-stopping patients as the 32-beat model
Xtr = np.concatenate([PREP16[r][0] for r in train]); ytr = np.concatenate([PREP16[r][1] for r in train])
Xva = np.concatenate([PREP16[r][0] for r in val]);   yva = np.concatenate([PREP16[r][1] for r in val])
cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * ytr.sum())}
tf.keras.backend.clear_session(); random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
m16 = build_rr_tcn(); m16.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
h16 = m16.fit(ds, validation_data=(Xva, yva), epochs=int(os.environ.get('EPOCHS', 40)), class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])
BLOB16 = to_int8(m16, Xtr); open(f'{OUT_DIR}/afib_rr_tcn16_int8.tflite', 'wb').write(BLOB16)
print('16-beat model: epochs', len(h16.history['loss']), '| windows', len(ytr), '| int8', len(BLOB16), 'bytes')
N_RR = 32

## Data: CinC 2017 (A and N) and NSTDB noise

In [ ]:
CINC_DIR = os.environ.get('CINC_DIR')
if not CINC_DIR:
    # the PhysioNet open-data mirror on AWS is much faster than physionet.org from Kaggle; same files
    MIRRORS = ['https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/',
               'https://physionet.org/files/challenge-2017/1.0.0/']
    os.makedirs('/kaggle/tmp', exist_ok=True)
    for f in ('REFERENCE-v3.csv', 'training2017.zip'):
        dst = f'/kaggle/tmp/{f}'
        for base in MIRRORS:
            try:
                t = time.time(); subprocess.run(['curl', '-fsSL', '--max-time', '1200', '-o', dst, base + f], check=True)
                print(f, os.path.getsize(dst) // 1024, 'KB from', base, f'in {time.time()-t:.0f} s', flush=True); break
            except Exception as e:
                print('mirror failed:', base, e, flush=True)
    zipfile.ZipFile('/kaggle/tmp/training2017.zip').extractall('/kaggle/tmp')
    CINC_DIR = '/kaggle/tmp/training2017'
    REF = '/kaggle/tmp/REFERENCE-v3.csv'
else:
    REF = os.path.join(CINC_DIR, 'REFERENCE-v3.csv')
ref = pd.read_csv(REF, header=None, names=['record', 'label'])
print(len(ref), 'recordings;', ref.label.value_counts().to_dict())


from scipy.signal import resample_poly as _rp
NS_DIR = os.environ.get('NS_DIR')
if not NS_DIR:
    NS_DIR = '/kaggle/tmp/nstdb'; os.makedirs(NS_DIR, exist_ok=True)
    for rec in ('em', 'ma'):
        for ext in ('hea', 'dat'):
            for base in ('https://physionet.org/files/nstdb/1.0.0/', 'https://physionet-open.s3.amazonaws.com/nstdb/1.0.0/'):
                if subprocess.run(['curl', '-fsSL', '--max-time', '600', '-o', f'{NS_DIR}/{rec}.{ext}', base + f'{rec}.{ext}']).returncode == 0: break
            else:
                raise RuntimeError(f'could not download NSTDB {rec}.{ext}')
NOISE300 = {}
for rec in ('em', 'ma'):
    r = wfdb.rdrecord(os.path.join(NS_DIR, rec), channels=[0]); NOISE300[rec] = _rp(r.p_signal[:, 0], 5, 6)  # 360 -> 300 Hz once
ALL = []
for rec, lab in zip(ref.record, ref.label):
    if lab in ('A', 'N'):
        r = wfdb.rdrecord(os.path.join(CINC_DIR, rec)); ALL.append((rec, lab, r.p_signal[:, 0], r.fs))
print(len(ALL), 'A/N recordings')

## (a) 16 vs 32 beats on clean recordings

In [ ]:
def decisions(blob, n_rr, rr):
    ks = list(range(0, len(rr) - n_rr + 1, DEV_STRIDE))
    if not ks: return None
    return run_tflite(blob, np.array([rr_features(rr[k:k + n_rr]) for k in ks], np.float32))

rows = []
for rec, lab, x, fs in ALL:
    b = detect_r(to_125(x, fs)); rr = np.diff(b) / FS
    p32, p16 = decisions(BLOB32, 32, rr), decisions(BLOB16, 16, rr)
    rows.append(dict(record=rec, label=lab, beats=len(b),
                     af32=None if p32 is None else float((p32 >= .5).mean() >= .5),
                     af16=None if p16 is None else float((p16 >= .5).mean() >= .5)))
W = pd.DataFrame(rows); W.to_csv(f'{OUT_DIR}/window16_vs_32.csv', index=False)

def sens_spec(d, col):
    d = d[d[col].notna()]; y = d.label == 'A'; p = d[col] == 1
    return dict(n_AF=int(y.sum()), n_N=int((~y).sum()), sens=float((p & y).sum() / y.sum()), spec=float((~p & ~y).sum() / (~y).sum()))
both = W[W.af32.notna() & W.af16.notna()]
only16 = W[W.af32.isna() & W.af16.notna()]
A = W.label == 'A'
R16 = {
  'coverage_32': float(W.af32.notna().mean()), 'coverage_16': float(W.af16.notna().mean()),
  'same_recordings_32': sens_spec(both, 'af32'), 'same_recordings_16': sens_spec(both, 'af16'),
  'only_16_can_classify': sens_spec(only16, 'af16') if len(only16) else None,
  'all_AF_found_32_unclassifiable_as_noAF': float((W[A].af32 == 1).sum() / A.sum()),
  'all_AF_found_16_unclassifiable_as_noAF': float((W[A].af16 == 1).sum() / A.sum()),
  'all_N_correct_32': float((W[~A].af32 != 1).mean()), 'all_N_correct_16': float((W[~A].af16 != 1).mean()),
}
print(json.dumps(R16, indent=2, default=lambda v: round(float(v), 4)))

## (b) Noise with and without the two-detector gate (32-beat model)

In [ ]:
from wfdb import processing
TOL = int(0.15 * FS); BSQI_MIN = 0.8

def bsqi(b1, b2):
    if len(b1) + len(b2) == 0: return 0.0
    j = np.searchsorted(b2, b1); m = 0
    for i, v in zip(j, b1):
        c = [b2[k] for k in (i - 1, i) if 0 <= k < len(b2)]
        if c and min(abs(v - u) for u in c) <= TOL: m += 1
    return m / (len(b1) + len(b2) - m)

def bandpass(x, fs): return sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), x)

def evaluate(x, fs):
    x125 = to_125(x, fs); b = detect_r(x125); rr = np.diff(b) / FS
    ks = list(range(0, len(rr) - 32 + 1, DEV_STRIDE))
    if not ks: return dict(n_dec=0)
    p = run_tflite(BLOB32, np.array([rr_features(rr[k:k + 32]) for k in ks], np.float32))
    with contextlib.redirect_stdout(io.StringIO()):
        b2 = np.sort(np.asarray(processing.xqrs_detect(sig=x125, fs=FS, verbose=False), dtype=int))
    keep = []
    for k in ks:
        end = b[k + 32]; a = max(0, end - 10 * FS)
        keep.append(bsqi(b[(b >= a) & (b < end)], b2[(b2 >= a) & (b2 < end)]) >= BSQI_MIN)
    keep = np.array(keep)
    out = dict(n_dec=len(ks), frac_af=float((p >= .5).mean()), n_keep=int(keep.sum()))
    if keep.any(): out['frac_af_gated'] = float((p[keep] >= .5).mean())
    return out

SIG = [(rec, lab, x, fs) for (rec, lab, x, fs), w in zip(ALL, W.itertuples()) if w.af32 is not None and not pd.isna(w.af32)]
print(len(SIG), 'recordings with >= 33 beats (same set as Notebook 4)')
CONDS = [('clean', None)] + [(k, s) for k in ('em', 'ma') for s in (12, 6, 0)]
rows, T0 = [], time.time()
for kind, snr in CONDS:
    rng = np.random.RandomState(SEED)
    for rec, lab, x, fs in SIG:
        if kind == 'clean':
            xn = x
        else:
            z = NOISE300[kind]; s = rng.randint(0, len(z) - len(x)); z = z[s:s + len(x)]
            ps, pn = bandpass(x, fs).var(), bandpass(z, fs).var()
            xn = x + z * np.sqrt(ps / (pn * 10 ** (snr / 10)) if pn > 0 else 0)
        rows.append(dict(record=rec, label=lab, noise=kind, snr=snr if snr is not None else 99, **evaluate(xn, fs)))
    print(f'{kind:5s} {snr}  [{time.time()-T0:.0f} s]', flush=True)
D = pd.DataFrame(rows); D.to_csv(f'{OUT_DIR}/bsqi_predictions.csv', index=False)

def metrics(g):
    y = (g.label == 'A').values; pred = (g.frac_af.fillna(0) >= .5).values
    has = g.frac_af_gated.notna().values; predg = (g.frac_af_gated.fillna(0) >= .5).values
    return pd.Series(dict(sens=pred[y].mean(), spec=(~pred[~y]).mean(),
                          sens_gated=predg[y & has].mean() if (y & has).any() else np.nan,
                          spec_gated=(~predg[~y & has]).mean() if (~y & has).any() else np.nan,
                          no_verdict_AF=(~has[y]).mean(), no_verdict_N=(~has[~y]).mean(),
                          false_AF_N=int(pred[~y].sum()), false_AF_N_gated=int((predg & has)[~y].sum()),
                          AF_found_gated_of_all_AF=(predg & has)[y].mean()))
T = D.groupby(['noise', 'snr']).apply(metrics).reset_index().sort_values(['noise', 'snr'], ascending=[True, False])
T.to_csv(f'{OUT_DIR}/bsqi_summary.csv', index=False)
pd.set_option('display.width', 220); print(T.round(3).to_string(index=False))
json.dump(dict(window16=R16, bsqi=T.round(4).to_dict(orient='records')), open(f'{OUT_DIR}/improve_summary.json', 'w'),
          indent=1, default=float)
print('total', round((time.time() - T0) / 60, 1), 'min')